# Domain Generalization

## Imports

In [1]:
import torch
import pandas as pd
import evaluate
from tqdm import tqdm
from datasets import Dataset
from rouge_score import rouge_scorer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

/opt/miniconda3/envs/bart_env_312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import CNN Model Result Summaries

In [9]:
model_summaries_df = pd.read_parquet("../data/models_summaries_table_cnn.parquet")
print(model_summaries_df.head())
model_summaries_dataset = Dataset.from_pandas(model_summaries_df)

                                              Source  \
0  for about 20 years the problem of properties o...   
1  it is believed that the direct detection of gr...   
2  as a common quantum phenomenon , the tunneling...   
3  for the hybrid monte carlo algorithm ( hmc)@xc...   
4  recently it was discovered that feynman integr...   

                                           Reference  \
0  Experts question if  packed out planes are put...   
1  Drunk teenage boy climbed into lion enclosure ...   
2  Nottingham Forest are close to extending Dougi...   
3  Fiorentina goalkeeper Neto has been linked wit...   
4  Tell-all interview with the reality TV star, 6...   

                                        BART Summary  \
0  U.S consumer advisory group set up by Departme...   
1  Rahul Kumar, 17, clambered over enclosure fenc...   
2  Dougie Freedman is on the verge of agreeing a ...   
3  Fiorentina goalkeeper Neto is wanted by a numb...   
4  The former Olympian and reality TV star, 65

## Score Each Model's Summary Compared to Reference

In [10]:
r_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
bert_scorer = evaluate.load("bertscore")

results = []

for sample in tqdm(model_summaries_dataset):
    source = sample['Source']
    reference = sample['Reference']

    bart_summary = sample['BART Summary']
    t5_summary = sample['T5 Summary']
    model_40k_summary = sample['Model 40k Summary']
    model_15k_summary = sample['Model 15k Summary']

    bart_scores = r_scorer.score(reference, bart_summary)
    t5_scores = r_scorer.score(reference, t5_summary)
    model_40k_scores = r_scorer.score(reference, model_40k_summary)
    model_15k_scores = r_scorer.score(reference, model_15k_summary)

    bart_bertscore = bert_scorer.compute(predictions=[bart_summary], references=[reference], lang="en")
    t5_bertscore = bert_scorer.compute(predictions=[t5_summary], references=[reference], lang="en")
    model_40k_bertscore = bert_scorer.compute(predictions=[model_40k_summary], references=[reference], lang="en")
    model_15k_bertscore = bert_scorer.compute(predictions=[model_15k_summary], references=[reference], lang="en")

    bart_cr = len(bart_summary.split()) / len(source.split())
    t5_cr = len(t5_summary.split()) / len(source.split())
    model_40k_cr = len(model_40k_summary.split()) / len(source.split())
    model_15k_cr = len(model_15k_summary.split()) / len(source.split())

    bart_density = bart_scores['rougeL'].fmeasure / (bart_cr + 1e-9)
    t5_density = t5_scores['rougeL'].fmeasure / (t5_cr + 1e-9)
    model_40k_density = model_40k_scores['rougeL'].fmeasure / (model_40k_cr + 1e-9)
    model_15k_density = model_15k_scores['rougeL'].fmeasure / (model_15k_cr + 1e-9)

    results.append({
        "bart_summary": bart_summary,
        "t5_summary": t5_summary,
        "model_40k_summary": model_40k_summary,
        "model_15k_summary": model_15k_summary,
        "bart_r1": bart_scores['rouge1'].fmeasure,
        "t5_r1": t5_scores['rouge1'].fmeasure,
        "model_40k_r1": model_40k_scores['rouge1'].fmeasure,
        "model_15k_r1": model_15k_scores['rouge1'].fmeasure,
        "bart_r2": bart_scores['rouge2'].fmeasure,
        "t5_r2": t5_scores['rouge2'].fmeasure,
        "model_40k_r2": model_40k_scores['rouge2'].fmeasure,
        "model_15k_r2": model_15k_scores['rouge2'].fmeasure,
        "bart_rL": bart_scores['rougeL'].fmeasure,
        "t5_rL": t5_scores['rougeL'].fmeasure,
        "model_40k_rL": model_40k_scores['rougeL'].fmeasure,
        "model_15k_rL": model_15k_scores['rougeL'].fmeasure,
        "bart_cr": bart_cr,
        "t5_cr": t5_cr,
        "model_40k_cr": model_40k_cr,
        "model_15k_cr": model_15k_cr,
        "bart_density": bart_density,
        "t5_density": t5_density,
        "model_40k_density": model_40k_density,
        "model_15k_density": model_15k_density
    })

df_results = pd.DataFrame(results)
df_results.head()

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 464.36it/s, Materializing param=encoder.layer.23.output.dense.weight]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 6000/6000 [2:13:13<00:00,  1.33s/it]     


,bart_summary,t5_summary,model_40k_summary,model_15k_summary,bart_r1,t5_r1,model_40k_r1,model_15k_r1,bart_r2,t5_r2,...,model_40k_rL,model_15k_rL,bart_cr,t5_cr,model_40k_cr,model_15k_cr,bart_density,t5_density,model_40k_density,model_15k_density
0,U.S consumer advisory group set up by Departme...,some experts say that the shrinking space on a...,the shrinking space on aeroplanes is putting o...,a U.S consumer advisory group set up by the De...,0.329670,0.252632,0.244604,0.285714,0.157303,0.043011,...,0.143885,0.197802,0.009427,0.010970,0.019541,0.009427,27.975222,17.271709,7.363372,20.981416
1,"Rahul Kumar, 17, clambered over enclosure fenc...","Rahul Kumar, 17, clambered over enclosure fenc...","Rahul Kumar , 17 , had to be rescued by secur...","Rahul Kumar, 17, climbed into the enclosure at...",0.571429,0.560000,0.483333,0.574074,0.341463,0.356164,...,0.333333,0.425926,0.007355,0.006283,0.014557,0.011033,64.742055,72.157387,22.898244,38.605449
2,Dougie Freedman is on the verge of agreeing a ...,Dougie Freedman is set to sign a new two-year ...,Dougie Freedman is set to sign a new two-year ...,Dougie Freedman is on the verge of agreeing a ...,0.366197,0.450704,0.459459,0.493506,0.144928,0.173913,...,0.243243,0.311688,0.014499,0.015707,0.016915,0.017318,15.543035,17.934271,14.380308,17.998187
3,Fiorentina goalkeeper Neto is wanted by a numb...,neto is wanted by a number of top european clu...,neto is wanted by a number of top European clu...,neto is wanted by a number of top European clu...,0.400000,0.395604,0.342342,0.368421,0.194175,0.247191,...,0.180180,0.192982,0.016963,0.014501,0.021067,0.019152,14.597541,15.156541,8.552708,10.076441
4,"The former Olympian and reality TV star, 65, w...",the 65-year-old will speak out in a 'far-rangi...,"the former Olympian and reality TV star , 65 ,...","the former Olympian and reality TV star, 65, w...",0.521739,0.474576,0.612903,0.648649,0.289308,0.189655,...,0.435484,0.608108,0.021053,0.012094,0.015006,0.018589,18.291925,18.218768,29.021423,32.713284


## Create Comparison Table

In [11]:
cnn_comparison_table = df_results[["bart_r1","bart_r2","bart_rL", "t5_r1", "t5_r2", "t5_rL","model_40k_r1","model_40k_r2","model_40k_rL","model_15k_r1","model_15k_r2","model_15k_rL","bart_cr","t5_cr","model_40k_cr","model_15k_cr","bart_density","t5_density","model_40k_density","model_15k_density"]].mean().to_frame(name="Average Scores")

display(cnn_comparison_table)

,Average Scores
bart_r1,0.442067
bart_r2,0.212169
bart_rL,0.309274
t5_r1,0.392971
t5_r2,0.168391
t5_rL,0.265278
model_40k_r1,0.377408
model_40k_r2,0.159014
model_40k_rL,0.243654
model_15k_r1,0.414285


## Save Scores

In [12]:
cnn_results_parquet_path = "../results/cnn_all_results.parquet"
df_results.to_parquet(cnn_results_parquet_path, index=False)

cnn_comparison_parquet_path = "../results/cnn_comparison_table.parquet"
cnn_comparison_table.to_parquet(cnn_comparison_parquet_path)

In [2]:
df_results = pd.read_parquet("../results/cnn_all_results.parquet")